# Figure 4B Clone Dominance Score Plot

This notebook minimally rebuilds only the clone-dominance score plot from the staged PyClone result tables plus the shared external clinical metadata file.


In [ ]:
from __future__ import annotations

import glob
import os
from pathlib import Path

os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib")
Path(os.environ["MPLCONFIGDIR"]).mkdir(parents=True, exist_ok=True)

import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

DEFAULT_NOTEBOOK_DIR = Path("/mnt/myvolume/PanelSeqMelanomaNotebook/analysis")
NOTEBOOK_DIR = Path(os.environ.get("PANELSEQ_NOTEBOOK_DIR", DEFAULT_NOTEBOOK_DIR)).expanduser().resolve()
PANEL_SEQ_ROOT = Path(os.environ.get("PANELSEQ_DATA_ROOT", "/mnt/myvolume/panel_seq")).expanduser().resolve()
CLINICAL_METADATA_PATH = Path(
    os.environ.get(
        "PANELSEQ_FIG1_CLINICAL_PATH",
        str(PANEL_SEQ_ROOT / "figure_inputs/figure1_clinical_minimal.tsv"),
    )
).expanduser().resolve()
PYCLONE_DIR = PANEL_SEQ_ROOT / "new_bed_analysis/clone_analysis/pyclone/pyclonevi_runs"

MIN_CLUSTER_MUTATIONS = 2
MIN_CCF_THRESHOLD = 0.05
CLONE_DOMINANCE_THRESHOLD = 0.85
MIN_SAMPLE_SNVS = 10

COLOR_PRIMARY = "#395f83"
COLOR_MET = "#9878ae"
plt.rcParams.update({
    "figure.dpi": 220,
    "savefig.dpi": 300,
    "font.family": "Liberation Sans",
    "font.size": 12,
    "axes.labelsize": 16,
    "xtick.labelsize": 12,
    "ytick.labelsize": 12,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})


def glob_count(pattern):
    return len(glob.glob(str(pattern), recursive=True))


def path_count(path_like):
    return int(Path(path_like).exists())


def build_required_inputs_table():
    rows = [
        ("Clinical Metadata", CLINICAL_METADATA_PATH, False, "external metadata", "paired primary/met cohort with matched normal codes"),
        ("PyClone result tables", PYCLONE_DIR / "*.results.tsv", True, "downstream output", "clone-level mutation clustering for primary/met pairs"),
    ]
    frame = pd.DataFrame(rows, columns=["input_family", "location", "is_glob", "category", "used_for"])
    frame["matches"] = frame.apply(lambda row: glob_count(row["location"]) if row["is_glob"] else path_count(row["location"]), axis=1)
    frame["present"] = frame["matches"] > 0
    return frame


def load_figure4_clinical_table(path=CLINICAL_METADATA_PATH):
    required = ["Sample", "patient_id", "paired_met_code", "normal_code", "figure3_paired_primary"]
    frame = pd.read_csv(path, sep="\t", dtype=str).fillna("")
    missing = [column for column in required if column not in frame.columns]
    if missing:
        raise ValueError(f"{path} is missing required Figure 4 clinical columns: {missing}")
    if frame["Sample"].duplicated().any():
        duplicates = sorted(frame.loc[frame["Sample"].duplicated(), "Sample"].unique())
        raise ValueError(f"Clinical metadata contains duplicated Sample IDs: {duplicates}")
    return frame.set_index("Sample", drop=False).sort_index()


def load_figure4_pairs(clinical_df):
    pair_df = (
        clinical_df.reset_index(drop=True)
        .loc[
            lambda df: (pd.to_numeric(df["figure3_paired_primary"], errors="coerce").fillna(0).astype(int) == 1)
            & df["paired_met_code"].ne("")
            & df["normal_code"].ne(""),
            ["patient_id", "Sample", "paired_met_code", "normal_code"],
        ]
        .rename(columns={"Sample": "primary_code", "paired_met_code": "met_code"})
        .drop_duplicates(subset=["patient_id", "primary_code", "met_code", "normal_code"])
        .sort_values(["patient_id", "primary_code", "met_code"])
        .reset_index(drop=True)
    )
    return pair_df


def parse_results_filename(path):
    name = Path(path).name
    if not name.endswith('.results.tsv'):
        return None
    core = name[:-len('.results.tsv')]
    if '__' not in core:
        return None
    return tuple(core.split('__', 1))


def index_pyclone_results(root=PYCLONE_DIR):
    available = {}
    for path in Path(root).glob('*.results.tsv'):
        parsed = parse_results_filename(path)
        if parsed:
            available[parsed] = path
    return available


def read_results(path):
    df = pd.read_csv(path, sep='\t', dtype=str, engine='python')
    df.columns = [column.strip().lower() for column in df.columns]
    return df


def mutation_key(df):
    for column in ['mutation_id', 'mutation', 'id', 'locus']:
        if column in df.columns:
            return df[column].astype(str)
    chrom = next((c for c in ['chromosome', 'chr', 'contig', 'chrom'] if c in df.columns), None)
    pos = next((c for c in ['position', 'pos', 'start'] if c in df.columns), None)
    ref = next((c for c in ['ref', 'reference'] if c in df.columns), None)
    alt = next((c for c in ['alt', 'alternate', 'mut', 'var', 'alt_allele'] if c in df.columns), None)
    if chrom and pos:
        refv = df[ref].astype(str) if ref else ''
        altv = df[alt].astype(str) if alt else ''
        return df[chrom].astype(str) + ':' + df[pos].astype(str) + ':' + refv + '>' + altv
    return pd.Series(df.index.astype(str))


def find_cluster_col(df):
    for column in ['cluster_id', 'cluster', 'cluster_assignment']:
        if column in df.columns:
            return column
    return None


def find_ccf_col(df):
    for column in ['cellular_prevalence', 'mean_cellular_prevalence', 'ccf', 'posterior_mean', 'mean']:
        if column in df.columns:
            return column
    return None


def count_real_clones(df, cluster_col, ccf_col):
    tmp = df[[cluster_col, ccf_col]].copy()
    tmp[cluster_col] = tmp[cluster_col].astype(str)
    tmp[ccf_col] = pd.to_numeric(tmp[ccf_col], errors='coerce').fillna(0.0)
    summary = (
        tmp.groupby(cluster_col, as_index=False)
        .agg(n_muts=(ccf_col, 'size'), mean_ccf=(ccf_col, 'mean'))
    )
    summary = summary[(summary['n_muts'] >= MIN_CLUSTER_MUTATIONS) & (summary['mean_ccf'] >= MIN_CCF_THRESHOLD)]
    return int(len(summary))


def calculate_clone_dominance(group):
    weights = (group['mean_ccf_m'] * group['n_shared_muts']).to_numpy(dtype=float)
    total = float(np.sum(weights))
    if total == 0:
        return 0.0
    dominance = float(np.max(weights) / total)
    burden_bonus = 0.1 * np.log10(total + 1)
    return float(dominance + burden_bonus)


def build_shared_cluster_stats(pair_df, available_results):
    pair_rows = []
    cluster_rows = []
    warnings = []
    for row in pair_df.itertuples(index=False):
        p_path = available_results.get((row.primary_code, row.normal_code))
        m_path = available_results.get((row.met_code, row.normal_code))
        if p_path is None or m_path is None:
            warnings.append((row.patient_id, 'missing_pyclone_file'))
            continue
        try:
            dfp = read_results(p_path)
            dfm = read_results(m_path)
        except Exception as exc:
            warnings.append((row.patient_id, f'read_error:{exc}'))
            continue
        cl_p = find_cluster_col(dfp)
        ccf_p = find_ccf_col(dfp)
        cl_m = find_cluster_col(dfm)
        ccf_m = find_ccf_col(dfm)
        if not (cl_p and ccf_p and cl_m and ccf_m):
            warnings.append((row.patient_id, 'missing_cluster_or_ccf_columns'))
            continue
        dfp = dfp.copy()
        dfm = dfm.copy()
        dfp['key'] = mutation_key(dfp)
        dfm['key'] = mutation_key(dfm)
        sub_p = dfp[[cl_p, 'key', ccf_p]].rename(columns={cl_p: 'cluster_id_primary', ccf_p: 'ccf_p'})
        sub_m = dfm[['key', ccf_m]].rename(columns={ccf_m: 'ccf_m'})
        merged = pd.merge(sub_p, sub_m, on='key', how='inner')
        merged['ccf_p'] = pd.to_numeric(merged['ccf_p'], errors='coerce').fillna(0.0)
        merged['ccf_m'] = pd.to_numeric(merged['ccf_m'], errors='coerce').fillna(0.0)
        shared = merged[(merged['ccf_p'] > MIN_CCF_THRESHOLD) & (merged['ccf_m'] > MIN_CCF_THRESHOLD)].copy()
        grouped = (
            shared.groupby('cluster_id_primary', as_index=False)
            .agg(n_shared_muts=('key', 'size'), mean_ccf_p=('ccf_p', 'mean'), mean_ccf_m=('ccf_m', 'mean'))
            if not shared.empty
            else pd.DataFrame(columns=['cluster_id_primary', 'n_shared_muts', 'mean_ccf_p', 'mean_ccf_m'])
        )
        robust = grouped[grouped['n_shared_muts'] >= MIN_CLUSTER_MUTATIONS].copy()
        pair_rows.append({
            'pt_code': row.patient_id,
            'primary_code': row.primary_code,
            'met_code': row.met_code,
            'normal_code': row.normal_code,
            'primary_mut_count': int(len(dfp)),
            'met_mut_count': int(len(dfm)),
            'n_primary_real_clones': count_real_clones(dfp, cl_p, ccf_p),
            'n_met_real_clones': count_real_clones(dfm, cl_m, ccf_m),
            'n_shared_muts_after_threshold': int(len(shared)),
            'n_robust_shared_clusters': int(len(robust)),
        })
        for cluster in robust.itertuples(index=False):
            cluster_rows.append({
                'pt_code': row.patient_id,
                'primary_code': row.primary_code,
                'met_code': row.met_code,
                'normal_code': row.normal_code,
                'cluster_id_primary': str(cluster.cluster_id_primary),
                'n_shared_muts': int(cluster.n_shared_muts),
                'mean_ccf_p': float(cluster.mean_ccf_p),
                'mean_ccf_m': float(cluster.mean_ccf_m),
            })
    return pd.DataFrame(pair_rows), pd.DataFrame(cluster_rows), warnings


def lighten(hex_color, amount=0.45):
    rgb = np.array(mcolors.to_rgb(hex_color))
    return tuple((1 - amount) * rgb + amount * np.array([1, 1, 1]))


In [ ]:
required_inputs = build_required_inputs_table()
display(Markdown('## Required staged inputs'))
display(required_inputs[['input_family', 'category', 'location', 'used_for', 'matches', 'present']])
missing_inputs = required_inputs.loc[~required_inputs['present']].copy()
if not missing_inputs.empty:
    raise FileNotFoundError(
        'Missing staged inputs for Figure 4 notebook:\n' + missing_inputs[['input_family', 'location']].to_string(index=False)
    )

clinical_df = load_figure4_clinical_table()
pair_df = load_figure4_pairs(clinical_df)
available_results = index_pyclone_results()
pair_stats_df, shared_cluster_df, pyclone_warnings = build_shared_cluster_stats(pair_df, available_results)

patient_scores = (
    shared_cluster_df.groupby('pt_code').apply(calculate_clone_dominance).reset_index(name='score')
    if not shared_cluster_df.empty
    else pd.DataFrame(columns=['pt_code', 'score'])
)
cohort_df = pair_stats_df.merge(patient_scores, on='pt_code', how='left').drop_duplicates(subset=['pt_code']).copy()
cohort_df['score'] = cohort_df['score'].fillna(0.0)
cohort_df['class'] = np.where(cohort_df['score'] < CLONE_DOMINANCE_THRESHOLD, 'polyclonal', 'monoclonal')
cohort_df['passes_min10'] = (
    (cohort_df['primary_mut_count'] >= MIN_SAMPLE_SNVS)
    & (cohort_df['met_mut_count'] >= MIN_SAMPLE_SNVS)
)
plot_df = cohort_df[cohort_df['score'] > 0].copy().sort_values(['score', 'pt_code'], ascending=[False, True]).reset_index(drop=True)
rng = np.random.default_rng(7)
plot_df['y'] = rng.normal(0.0, 0.19, size=len(plot_df))

cohort_summary = pd.DataFrame([
    {'metric': 'paired primaries in external clinical metadata', 'value': len(pair_df)},
    {'metric': 'pairs with PyClone files and usable columns', 'value': len(cohort_df)},
    {'metric': f'threshold monoclonal pairs (score >= {CLONE_DOMINANCE_THRESHOLD})', 'value': int((cohort_df['class'] == 'monoclonal').sum())},
    {'metric': f'pairs passing min {MIN_SAMPLE_SNVS} PyClone SNVs in both primary and met', 'value': int(cohort_df['passes_min10'].sum())},
    {'metric': 'pairs shown in clone-dominance score plot', 'value': len(plot_df)},
])

display(Markdown('## Figure 4 score cohort'))
display(cohort_summary)
display(
    cohort_df[['pt_code', 'primary_code', 'met_code', 'normal_code', 'score', 'class', 'primary_mut_count', 'met_mut_count', 'n_primary_real_clones', 'n_met_real_clones', 'n_shared_muts_after_threshold', 'n_robust_shared_clusters', 'passes_min10']]
    .sort_values(['score', 'pt_code'], ascending=[False, True])
    .reset_index(drop=True)
)
if pyclone_warnings:
    print('Pairs skipped during PyClone preprocessing:', len(pyclone_warnings))


In [ ]:
def draw_clone_dominance_score_panel(ax, plot_df):
    transition_cmap = mcolors.LinearSegmentedColormap.from_list(
        'poly_to_mono_transition',
        [COLOR_PRIMARY, lighten(COLOR_PRIMARY, 0.45), lighten(COLOR_MET, 0.45), COLOR_MET],
    )
    xmin = float(plot_df['score'].min()) if not plot_df.empty else 0.0
    xmax = float(plot_df['score'].max()) if not plot_df.empty else 1.0
    pad = max(0.03, 0.04 * (xmax - xmin)) if xmax > xmin else 0.03
    xleft, xright = xmin - pad, xmax + pad

    gradient = np.linspace(0, 1, 512).reshape(1, -1)
    ax.imshow(
        gradient,
        extent=[xleft, xright, -0.55, 0.55],
        cmap=transition_cmap,
        aspect='auto',
        alpha=0.28,
        zorder=1,
    )
    score_norm = np.clip((plot_df['score'] - xleft) / (xright - xleft), 0, 1) if len(plot_df) else np.array([])
    point_colors = transition_cmap(score_norm) if len(plot_df) else []
    ax.scatter(
        plot_df['score'],
        plot_df['y'],
        s=55,
        c=point_colors,
        edgecolors='black',
        linewidths=0.8,
        alpha=0.95,
        zorder=3,
    )

    ax.annotate('Polyclonal', xy=(0.14, 1.08), xycoords='axes fraction', ha='center', va='center', fontsize=14)
    ax.annotate('', xy=(0.20, 1.08), xycoords='axes fraction', xytext=(0.80, 1.08), textcoords='axes fraction', arrowprops=dict(arrowstyle='<->', lw=1.6, color='black'))
    ax.annotate('Monoclonal', xy=(0.86, 1.08), xycoords='axes fraction', ha='center', va='center', fontsize=14)

    ax.set_xlim(xleft, xright)
    ax.set_ylim(-0.55, 0.55)
    ax.set_yticks([])
    ax.set_ylabel('')
    ax.set_xlabel('Clone Dominance Score')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.grid(axis='x', linestyle=':', linewidth=0.8, alpha=0.25)
    ax.grid(axis='y', visible=False)


display(Markdown('## Figure 4B: clone-dominance score plot'))
print('Scored pairs shown in the plot:', len(plot_df))

fig, ax = plt.subplots(figsize=(7.2, 3.2), dpi=220)
draw_clone_dominance_score_panel(ax, plot_df)
fig.subplots_adjust(left=0.08, right=0.98, top=0.86, bottom=0.18)
plt.show()
